# A New Taxonomy For Attributed Graph Missingness Mechanisms

This jupyter notebook contains the experiments illustrating the paper "Throw off the Masks! Defining a Taxonomy for Masking Mechanisms in Attributed Graphs".

- In a first time, we download the relevant datasets: Planetoid (CORA, CITESEER, PUBMED), WebKB (CORNELL, TEXAS, WISCONSIN), etc. 
- In a second time we implement the different missingness mechanisms: MCAR, MAR, MNAR, through the new definitions proposed in the paper.
- In a third time, we implement the different imputation strategies: mean, median, mode, KNN, OT-tab (Muzellec et al.), FP (Feature Propagation - Rossi et al.), GRIOT (Graph Imputation with Optimal Transport - Serrano et al.), PCFI (PC-based Feature Imputation - Umet et al.), etc.
- In a fourth time, we evaluate the imputation strategies on a subset of all possible scenarios, with the different datasets, missingness mechanisms and imputation methods.

We expect to see a drop in imputation performance when the missingness mechanisms become complex, leveraging the graph's structure. Even if the imputation methods tailored for graph should exceed the performance of the tabular ones, we expect to see a drop in performance when the missingness mechanisms become complex.

# imports

In [ ]:
# general imports
import os
import sys
import time
from tqdm.auto import tqdm

# data manipulation
import numpy as np
import pandas as pd
import networkx as nx

In [ ]:
path = "./a_new_taxonomy_for_attributed_graph_missingness_mechanisms"

# Ensure the results directory exists
if not os.path.exists(path):
    os.makedirs(path)

# Load the datasets

- Planetoid: 
    - CORA, CITESEER, PUBMED
    - Citation networks
    - High homophily
    - Binary features (except for PUBMED)
- WebKB:
    - CORNELL, TEXAS, WISCONSIN
    - Web pages
    - Low homophily
    - Binary features

- [HeterophilousGraphDataset](https://pytorch-geometric.readthedocs.io/en/latest/generated/torch_geometric.datasets.HeterophilousGraphDataset.html) 
    - *Platonov et al. (2023)* -> response to the limitations in existing heterophily benchmarks. 
    - Goal: provide a diverse set of graphs that:
        - Display varying degrees of heterophily
        - Feature non-binary node attributes (including real and categorical features)
        - Come from a range of domains with varied structural properties
    - Datasets:
        - Minesweeper:
            - A synthetic dataset inspired by the classic Minesweeper game, built as a regular 100×100 grid.
            - NODES: Each node in the graph corresponds to a cell in the grid. 20% of the nodes are randomly selected as mines.
            - EDGES: Two cells are connected with an edge if they are adjacent to each other. Each node (cell) is connected to eight neighboring nodes.
            - FEATURES: The node features are one-hot-encoded numbers of neighboring mines. However, for randomly selected 50% of the nodes, the features are unknown, which is indicated by a separate binary feature.
            - CLASSES: The class of a node is the presence of a mine in the cell.
        - Tolokers:
            - Based on real data from the Toloka crowdsourcing platform, this dataset captures interactions among users engaged in tasks.
            - NODES: The nodes represent tolokers (workers) that have participated in at least one of 13 selected projects.
            - EDGES: Two users are connected with an edge if they have worked on the same task.
            - FEATURES: Node features are based on the worker’s profile information and task performance statistics.
            - CLASSES: The goal is to predict which tolokers have been banned in one of the projects.
|                | roman-empire | amazon-ratings | minesweeper | tolokers | questions |
|----------------|---------------|-----------------|--------------|----------|-----------|
| nodes          | 22662         | 24492           | 10000        | 11758    | 48921     |
| edges          | 32927         | 93050           | 39402        | 519000   | 153540    |
| avg degree     | 2.91          | 7.60            | 7.88         | 88.28    | 6.28      |
| global clustering | 0.29          | 0.32            | 0.43         | 0.23     | 0.02      |
| avg local clustering | 0.39          | 0.58            | 0.44         | 0.53     | 0.03      |
| diameter       | 6824          | 46              | 99           | 11       | 16        |
| node features  | 300           | 300             | 7            | 10       | 301       |
| classes        | 18            | 5               | 2            | 2        | 2         |
| edge homophily | 0.05          | 0.38            | 0.68         | 0.59     | 0.84      |
| adjusted homophily | -0.05         | 0.14            | 0.01         | 0.09     | 0.02      |
| label informativeness | 0.11          | 0.04            | 0.00         | 0.01     | 0.00      |


# Missingness Mask Generator - Graph Structure Aware

Below is a class that implements missingness mask generation for graph data (i.e. on the node feature matrix) according to three mechanisms: MCAR, MAR, and MNAR.

There are more mechanisms than with tabular data, as we can leverage the graph structure to create more complex missingness mechanisms.

## Aggregator function

Below are 4 aggregator functions that can be used to generate missingness masks for graph data.

### 1. MAR – Scenario 2: Only Structural Properties

_In this scenario, the missingness depends solely on structural properties; here we use the node degree as a proxy._
For each node $ v_i $ we define
$$
\tilde{d}_i = \frac{d_i}{\max\limits_{k} d_k},
$$
where $ d_i $ is the degree of node $ v_i $. Then, for every feature $j$ of $v_i$ the missingness probability is given by
$$
p_{ij} = \sigma\Big(\alpha\, (\tilde{d}_i - 0.5)\Big),
$$
with $\alpha=5$ and $\sigma$ the sigmoid function. Notice that the same probability is applied across all features for node $v_i$.

### 2. MAR – Scenario 3: Neighbors’ Observed Features Dominate

_Here the missingness probability for a node depends on both its own observed features and on the observed features in its neighborhood—with a higher weight on the neighbors’ statistics._

Let
- $ F^{(OBS)} $ be a subset of $F$ (column-wise) such that all features in $F^{(OBS)}$ are fully observed.
- $ F^{(MIS)}=F\subset F^{(OBS)} $ in contrast is the subset of potentially unobserved features.
- $ F^{(\text{self})}_i $ be the mean of the observed features of node $v_i$, and
- $ F^{(\text{nbr})}_i $ the average of the observed features of its neighbors.

Then we define for each node $v_i$ and every feature $j$:
$$
p_{ij} = \sigma\Big(\beta_1 \Big(F^{(\text{nbr})}_i - \theta\Big) + \beta_2 \Big(F^{(\text{self})}_i - \theta\Big)\Big),
$$
with $\beta_1=3$, $\beta_2=1$, and $\theta=0.5$. Again, the same probability is applied to every feature for a given node.


### 3. MNAR – Scenario 3: Neighbors’ Unobserved Features Dominate

_In this scenario, missingness is driven by the unobserved (i.e. potentially masked) features. For each node $v_i$ and feature $j$, we assume that both the node’s own value and the average of its neighbors’ values (for the same feature) contribute to the probability. Since we are in an MNAR context, the calculation is done for each feature independently._

Let
- $ F_{ij} $ be the feature value of node $v_i$ at feature $j$ (the “self” term), and
- $ \overline{F}^{(\text{nbr})}_{ij} $ be the average of the corresponding feature $j$ over all neighboring nodes.

Then we set:
$$
p_{ij} = \sigma\Big(\gamma_1\Big( F_{ij} - 0.5\Big) + \gamma_2\Big( \overline{F}^{(\text{nbr})}_{ij} - 0.5\Big) \Big),
$$
where we choose, for example, $\gamma_1=1$ and $\gamma_2=3$ (so that neighbors receive more influence).

### 4. MNAR – Scenario 4: Balanced Node and Neighbor Information

_In this final MNAR scenario, we assume that missingness depends on a balance of three factors: (i) the node’s own value, (ii) its structural information (here through its normalized degree), and (iii) the average feature value of its neighbors. For each node $v_i$ and feature $j$ we define:_

First, compute the normalized degree:
$$
\tilde{d}_i = \frac{d_i}{\max_k d_k}.
$$
Then, using weights $\delta_1$, $\delta_2$, and $\delta_3$ (here all set to 1 for simplicity), define
$$
p_{ij} = \sigma\Big( \delta_1\Big( F_{ij} - 0.5\Big) + \delta_2\Big( \tilde{d}_i - 0.5\Big) + \delta_3\Big( \overline{F}^{(\text{nbr})}_{ij} - 0.5\Big) \Big).
$$


# Imputation


In [ ]:
# Import the core experiment function and debug printer
from a_new_taxonomy_for_attributed_graph_missingness_mechanisms_RUN import run_experiments_with_config, print_debug_info


def notebook_run(
        datasets=["Texas"],  # List of datasets to use
        models=None,  # List of imputer models (None = all models)
        missing_rates=(0.2,),  # Tuple of missing rates
        output_path="./a_new_taxonomy_for_attributed_graph_missingness_mechanisms",
        runs=1,  # Number of experiment runs
        start=0,  # Index at which exp starts
        keep_main_component=True,  # Whether to keep only the main connected component
        verbose=True  # Print debug information
):
    """
    Run experiments from a Jupyter notebook.

    This function prepares a configuration and calls the same core function
    used by the command-line script.
    """
    # Validate datasets
    valid_datasets = ["Cora", "CiteSeer", "PubMed", "Texas", "Wisconsin",
                      "Cornell", "Minesweeper", "Tolokers", "Questions"]
    selected_datasets = [d for d in datasets if d in valid_datasets]

    if not selected_datasets:
        print("No valid datasets specified. Using default 'Texas'.")
        selected_datasets = ["Texas"]

    # Validate imputers if specified
    imputer_names = models
    if imputer_names:
        valid_imputers = ["Tabular_Avg", "Random", "Graph_1hop", "MICE",
                          "FP", "OT-tab", "PCFI", "GRIOT"]
        imputer_names = [m for m in imputer_names if m in valid_imputers]

        if not imputer_names:
            print("No valid imputers specified. Using all available imputers.")
            imputer_names = None

    # Create the configuration dictionary
    config = {
        "selected_datasets": selected_datasets,
        "imputer_names": imputer_names,
        "missing_rates": missing_rates,
        "output_path": output_path,
        "runs": runs,
        "start": start,
        "keep_main_component": keep_main_component,
        "verbose": verbose
    }

    # Print debug information if requested
    if verbose:
        print_debug_info(config)

    # Run the experiments using the shared function
    return run_experiments_with_config(config)

if __name__ == "__main__":
    # Example: Run a simple test
    results = notebook_run(
        datasets=["Cornell"],
        models=["FP"],
        missing_rates=(0.2,),
        runs=1
    )